In [0]:
import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.ensemble import IsolationForest
from pyspark.sql import functions as F
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Parametros

In [0]:
ambiente = 'dev'
costa = 'Matamoros'
DATOS_PORCENTAJE = 1

FEATURES_SCANER = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_steepness'
]
if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    
ISO_FOREST_PORCENTAJE = 1
RANDOM_SEED = 0
N_ESTIMATORS = 100
MAX_SAMPLES = 0.7
CONTAMINATION = 0.05

# Obtener datos

In [0]:
data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, wind_speed_ms, wind_cos_direction, wind_sin_direction, 
                wave_height_m, wave_cos_direction, wave_sin_direction, wave_period_s, wave_energy, wave_steepness
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{costa}'
            """
        )
    )

In [0]:
data = (
    data
    .withColumn('coast_year_month', F.concat(F.col('coast_name'), F.lit('_'), F.date_format('datetime', 'yyyy-MM')))
)

In [0]:
coast_year_month_distinct = data.select('coast_year_month').distinct().collect()

In [0]:
fig = px.scatter_3d(
    data.toPandas(), x='wind_speed_ms',
    y='wave_height_m',
    z='wave_period_s'
)
fig.show()

# Escalar